<a href="https://colab.research.google.com/github/25031554180-bit/Feature-Engineering/blob/main/Aliya(030)Restika(180)feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



```
# This is formatted as code
```

*   Aliya Munafi'ah(25031554030)
*   Restika Zakiyatus Syahida (25031554180)



In [ ]:
# 1. SCRAPING DATA BUKU (TUGAS SEBELUMNYA)
import requests
from bs4 import BeautifulSoup
import pandas as pd

root = 'https://books.toscrape.com/catalogue/page-'
titles = []
prices = []
ratings = []

# Scrape 5 halaman pertama
for page in range(1, 6):
    url = requests.get(root + str(page) + '.html')
    soup = BeautifulSoup(url.text, 'html.parser')
    scrap = soup.find_all('article', {'class': 'product_pod'})

    for i in scrap:
        titles.append(i.find('h3').find('a')['title'])
        prices.append(i.find('p', {'class': 'price_color'}).text)
        ratings.append(i.find('p', {'class': 'star-rating'})['class'][1])

# Buat DataFrame awal
df = pd.DataFrame({
    'Title': titles,
    'Price': prices,
    'Rating': ratings
})



In [ ]:
# 2. PRE-PROCESSING (CHAPTER 4)
!pip install demoji
import re
import string
import demoji
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

def remove_html(text):
    return re.sub(r'<.*?>', '', str(text))

def remove_hashtag(text):
    return re.sub(r'#\w+', '', str(text))

def remove_rls(text):
    text = re.sub(r'https?://\S+', '', str(text))
    text = re.sub(r'http\S+', '', str(text))
    text = re.sub(r'www\S+', '', str(text))
    text = re.sub(r'\S+@\S+', '', str(text))
    return text

def remove_punctuation_list(text):
    text = re.sub(r'(?::|;|=)(?:-)?(?:\)|D|\||\(|\\|\/)', '', str(text))
    text = re.sub(r'[!"#$%&\'()*+,-./:;<=>?@\[\\\]^_`{|}~]', ' ', str(text))
    return re.sub(r'\s+', ' ', str(text)).strip()

def remove_emoji(text):
    return demoji.replace(str(text), repl="")

sw = set(stopwords.words('english'))
def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in sw])

ps = PorterStemmer()
def stem_text(text):
    return " ".join([ps.stem(word) for word in str(text).split()])

# Jalankan Preprocessing
df['rm_html'] = df['Title'].apply(remove_html)
df['rm_hashtag'] = df['rm_html'].apply(remove_hashtag)
df['lower_case'] = df['rm_hashtag'].str.lower()
df['rm_url'] = df['lower_case'].apply(remove_rls)
df['rm_punc'] = df['rm_url'].apply(remove_punctuation_list)
df['rm_emoji'] = df['rm_punc'].apply(remove_emoji)
df['stem'] = df['rm_emoji'].apply(stem_text)
df['prepro'] = df['stem'].apply(remove_stopwords)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 3.1 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Feature Engineering part 1

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
df_pre = df['prepro'].dropna().reset_index().drop(columns=['index'])['prepro']

Transformasi TF-IDF

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df_pre)
df_FE = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

print("--- Hasil TF-IDF ---")
print(df_FE.head())

--- Hasil TF-IDF ---
    01   12   15   16  1800  1849  1883  1936  1981  1991  ...  wilder  \
0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...     0.0   
1  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...     0.0   
2  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...     0.0   
3  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...     0.0   
4  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...     0.0   

   without  woman  wonder  wood  woodhul  work  workbook  world  worri  
0      0.0    0.0     0.0   0.0      0.0   0.0       0.0    0.0    0.0  
1      0.0    0.0     0.0   0.0      0.0   0.0       0.0    0.0    0.0  
2      0.0    0.0     0.0   0.0      0.0   0.0       0.0    0.0    0.0  
3      0.0    0.0     0.0   0.0      0.0   0.0       0.0    0.0    0.0  
4      0.0    0.0     0.0   0.0      0.0   0.0       0.0    0.0    0.0  

[5 rows x 385 columns]


Feature Engineering part 2

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

Tokeniza Sentence

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from nltk import word_tokenize
from tqdm import tqdm

sentences = [word_tokenize(tweet.lower()) for tweet in tqdm(df_pre)]
sentences

100%|██████████| 100/100 [00:00<00:00, 4767.72it/s]


[['light', 'attic'],
 ['tip', 'velvet'],
 ['soumiss'],
 ['sharp', 'object'],
 ['sapien', 'brief', 'histori', 'humankind'],
 ['requiem', 'red'],
 ['dirti', 'littl', 'secret', 'get', 'dream', 'job'],
 ['come',
  'woman',
  'novel',
  'base',
  'life',
  'infam',
  'feminist',
  'victoria',
  'woodhul'],
 ['boy',
  'boat',
  'nine',
  'american',
  'epic',
  'quest',
  'gold',
  '1936',
  'berlin',
  'olymp'],
 ['black', 'maria'],
 ['starv', 'heart', 'triangular', 'trade', 'trilog'],
 ['shakespear', 'sonnet'],
 ['set', 'free'],
 ['scott', 'pilgrim', 'preciou', 'littl', 'life', 'scott', 'pilgrim'],
 ['rip', 'start'],
 ['band',
  'could',
  'life',
  'scene',
  'american',
  'indi',
  'underground',
  '1981',
  '1991'],
 ['olio'],
 ['mesaerion', 'best', 'scienc', 'fiction', 'stori', '1800', '1849'],
 ['libertarian', 'beginn'],
 ['onli', 'himalaya'],
 ['wake'],
 ['music', 'work'],
 ['foolproof',
  'preserv',
  'guid',
  'small',
  'batch',
  'jam',
  'jelli',
  'pickl',
  'condiment',
  'foo

Build Word2vec Model

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 26.5 MB/s eta 0:00:00


In [ ]:
import gensim.models

In [ ]:
model = gensim.models.Word2Vec(sentences, min_count=1, vector_size=100, window=5, sg=0)

In [ ]:
model.save('scraped_books.w2v')

In [ ]:
model = gensim.models.Word2Vec.load('scraped_books.w2v')

In [ ]:
w2v = model.wv
w2v.index_to_key[:10]

['life',
 'lumberjan',
 '1',
 'vol',
 'collect',
 'world',
 'guid',
 'princess',
 'saga',
 'art']

In [ ]:
w2v.vectors.shape, w2v.vector_size # size of vector

((392, 100), 100)

In [ ]:
w2v = model.wv
sample_word = w2v.index_to_key[0]
print(f"\nKata yang serupa dengan '{sample_word}':")
print(w2v.similar_by_key(sample_word))


Kata yang serupa dengan 'life':
[('object', 0.36081933975219727), ('littl', 0.29208165407180786), ('draw', 0.27485910058021545), ('fine', 0.27354350686073303), ('tattoo', 0.2297949641942978), ('labor', 0.22110913693904877), ('citi', 0.22027966380119324), ('art', 0.21697606146335602), ('girl', 0.2083243429660797), ('wake', 0.2056797295808792)]


In [ ]:
import numpy as np

def get_document_vector(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not valid_vectors:
        return np.zeros(model.vector_size)
    return np.mean(valid_vectors, axis=0)

X_features = np.array([get_document_vector(doc, model) for doc in sentences])
df_w2v_features = pd.DataFrame(X_features)

print("\n--- Hasil Fitur Word2Vec ---")
print(df_w2v_features.head())


--- Hasil Fitur Word2Vec ---
         0         1         2         3         4         5         6   \
0 -0.008258  0.006298 -0.002986 -0.001885 -0.005149  0.003353  0.004550   
1 -0.003236  0.008996  0.004869 -0.005253 -0.003173  0.002976  0.001306   
2  0.005259  0.008041  0.006036 -0.001259 -0.002416 -0.004594 -0.009196   
3 -0.004808 -0.000687  0.005629 -0.007170 -0.000896 -0.002407  0.001055   
4  0.000193 -0.001851  0.000528 -0.000858 -0.001663 -0.004402 -0.002035   

         7         8         9   ...        90        91        92        93  \
0  0.004346  0.006418  0.001122  ... -0.001844 -0.003395  0.001211  0.002661   
1  0.004804  0.003960 -0.002099  ... -0.009013  0.003184  0.007530 -0.006789   
2 -0.003120 -0.005313 -0.001036  ...  0.009515  0.003538 -0.003111 -0.003635   
3  0.007257  0.004030  0.000368  ... -0.001987 -0.003048  0.003033 -0.001915   
4  0.002076 -0.001851  0.004676  ...  0.000029  0.001249 -0.004261  0.000360   

         94        95        96       

Visualize

In [ ]:
!pip install umap-learn

In [ ]:
from umap import UMAP

In [ ]:
import numpy as np
import plotly.express as px

In [ ]:
X = UMAP().fit_transform(w2v.vectors)

df2 = pd.DataFrame(X, columns=['umap1', 'umap2'])
df2['word'] = w2v.index_to_key

fig = px.scatter(df2, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height=800,
                  title_text='Word2Vec Visualization')
fig.show()

In [ ]:
import gensim.models
#build model
model_fasttext = gensim.models.FastText(sentences, min_count=1,
                                        vector_size=100, window=5,
                                        min_n=3, max_n=6,
                                        sg=1, epochs=10)

In [ ]:
# 2. save model
model_fasttext.save('scraped_books.fasttext')

In [ ]:
# 3. load model
model_fasttext = gensim.models.Word2Vec.load('scraped_books.fasttext')

In [ ]:
# 4. model information
fastext = model_fasttext.wv
fastext.index_to_key[:10]

['life',
 'lumberjan',
 '1',
 'vol',
 'collect',
 'world',
 'guid',
 'princess',
 'saga',
 'art']

In [ ]:
fastext.vectors.shape

(392, 100)

In [ ]:
fastext.similar_by_key('4g')

[('electr', 0.2962241768836975),
 ('wanderlov', 0.2867061197757721),
 ('belt', 0.2458343207836151),
 ('15', 0.23726697266101837),
 ('throw', 0.22636079788208008),
 ('track', 0.21728269755840302),
 ('collect', 0.2133502960205078),
 ('day', 0.1855967491865158),
 ('black', 0.175802081823349),
 ('job', 0.17219266295433044)]

In [ ]:
# 5. FE result
import numpy as np
def get_document_vector(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    # if not valid_vectors:
    #   return np.zeros(model.vector_size)
    return np.mean(valid_vectors, axis=0)
X_features = np.array([get_document_vector(doc, model_fasttext) for doc in sentences])
pd.DataFrame(X_features)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.000984,0.001109,-0.000261,-0.000236,-0.000181,0.000117,0.000328,0.001797,0.001590,-0.000303,...,0.000189,-0.000090,-0.001892,-0.001600,0.001597,-0.000174,0.000702,-0.001756,0.000877,0.000195
1,-0.001798,-0.000158,0.000740,0.003495,-0.000926,-0.001768,0.001846,0.003670,-0.000526,-0.000232,...,-0.001604,0.002620,0.002086,-0.000364,-0.001278,0.002060,-0.000544,0.000560,0.002381,0.000382
2,0.001059,0.000820,-0.000682,-0.000722,-0.002221,0.000835,0.000173,0.000291,0.002297,-0.000365,...,0.001990,-0.001125,-0.000427,-0.000324,0.002105,0.001555,-0.001605,0.000767,-0.001073,0.000610
3,0.000184,0.002499,-0.000520,0.000988,0.000177,-0.000058,0.000168,0.001439,0.000068,0.000420,...,0.001713,0.000252,-0.000442,0.000213,-0.000380,0.001754,0.000160,-0.002603,0.000656,-0.000793
4,-0.000644,-0.000560,-0.000465,0.000264,-0.000362,-0.000271,-0.000391,0.000195,0.000363,-0.000076,...,-0.000537,0.000553,-0.000248,-0.000734,0.000548,-0.000018,-0.000757,0.001117,0.000471,0.001188
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-0.001756,0.000144,0.000555,0.000129,0.000149,-0.000706,0.000521,0.001116,0.000325,-0.000126,...,-0.000711,0.000120,-0.001484,-0.000862,-0.000586,-0.000427,0.000917,-0.000501,0.001307,0.001020
96,0.000267,0.001303,0.001059,0.000167,0.000382,-0.000614,0.000060,-0.000267,0.000817,-0.000176,...,-0.001069,0.000722,-0.000767,0.000389,-0.000016,0.000344,-0.000010,-0.000283,0.000988,0.000668
97,-0.001411,0.000215,-0.000001,-0.000587,0.000309,-0.000232,-0.001679,-0.000348,0.000099,-0.000496,...,-0.000550,-0.000113,0.000150,0.000029,-0.000888,0.000349,0.000453,-0.000800,0.001321,0.000455
98,-0.002548,0.000817,0.000518,-0.003088,0.001152,0.000351,0.001526,0.000592,0.000956,0.003227,...,-0.003947,0.000506,-0.002167,0.000838,0.001684,-0.001418,-0.000789,0.000514,0.001999,-0.000855


In [ ]:
# Visualize
X = UMAP().fit_transform(fastext.vectors)

df2 = pd.DataFrame(X, columns=['umap1', 'umap2'])
df2['word'] = fastext.index_to_key

fig = px.scatter(df2, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height = 800,
                  title_text = 'fastext Visualization')
fig.show()